# 01. Chuẩn bị môi trường và chạy thử 1 epoch

**Trước khi chạy:** bên phải chọn *Settings*: Accelerator = GPU (P100 hoặc T4), Internet = On.
Bấm *Add Input* và thêm bộ dữ liệu `xxdxdxd/seedlingstemrow`.

Notebook này thực hiện: tải mã nguồn, vá repo gốc, chuẩn bị dữ liệu, ghi thông tin môi trường, kiểm tra repo để chốt baseline, rồi chạy thử 1 epoch cho từng cấu hình để chắc chắn không lỗi trước khi chạy dài.

In [ ]:
# Bước 1: tải mã nguồn của bạn từ GitHub (sửa GITHUB_USER cho đúng, repo phải để chế độ Public)
import os, sys, subprocess
GITHUB_USER = "TEN_GITHUB_CUA_BAN"
TEN_REPO = "ssr-btl-thigiacmay"
DUONG_DAN = f"/kaggle/working/{TEN_REPO}"
if not os.path.isdir(DUONG_DAN):
    subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{GITHUB_USER}/{TEN_REPO}.git", DUONG_DAN], check=True)
else:
    subprocess.run(["git", "-C", DUONG_DAN, "pull"], check=True)   # lấy bản mới nhất sau mỗi lần bạn push
sys.path.insert(0, f"{DUONG_DAN}/src")
print("Mã nguồn sẵn sàng:", sorted(os.listdir(f"{DUONG_DAN}/src")))

In [ ]:
# Bước 2: tải repo gốc YOLO-SSR và áp dụng các bản vá tương thích (an toàn khi chạy lại)
import setup_env
setup_env.cai_dat()

In [ ]:
# Bước 3: chuẩn bị dữ liệu. Lần đầu sẽ thu nhỏ ảnh (có thể mất vài chục phút); in ra số ảnh mỗi tập để bạn đối chiếu với dữ liệu gốc.
import ssr_lib as L
L.chuan_bi_du_lieu()

In [ ]:
# Bước 4: ghi phiên bản Python, thư viện, GPU và commit của repo gốc (bằng chứng tái lập cho báo cáo)
L.ghi_moi_truong()

In [ ]:
# Bước 5: kiểm tra repo để chốt baseline. Gửi toàn bộ kết quả của cell này cho trợ lý.
import os, re, glob
REPO = L.REPO
print("=== CÁC FILE YAML MÔ HÌNH ===")
for f in sorted(glob.glob(f"{REPO}/**/*.yaml", recursive=True)):
    if any(k in f for k in ["models", "cfg"]) and "/datasets/" not in f:
        print(" ", os.path.relpath(f, REPO), f"({os.path.getsize(f)} byte)")
print("\n=== NỘI DUNG YOLO-SSR.yaml ===")
for f in glob.glob(f"{REPO}/**/YOLO-SSR.yaml", recursive=True):
    print(f"# {os.path.relpath(f, REPO)}")
    print(open(f, encoding="utf-8").read())
print("\n=== README: CÁC DÒNG LIÊN QUAN ===")
for f in glob.glob(f"{REPO}/README*"):
    for i, d in enumerate(open(f, encoding="utf-8", errors="ignore").read().splitlines(), 1):
        if re.search(r"baseline|ablation|compar|YOLOv8|YOLOP|params|GFLOPs|mAP|weights|pretrain", d, re.I):
            print(f"  [{i}] {d.strip()[:200]}")
print("\n=== FILE TRỌNG SỐ (.pt) TRONG REPO ===")
for f in glob.glob(f"{REPO}/**/*.pt", recursive=True):
    print(" ", os.path.relpath(f, REPO), f"({os.path.getsize(f)/1e6:.1f} MB)")

In [ ]:
# Bước 6: chạy thử 1 epoch với mô hình chính (E_main) để kiểm tra toàn bộ chuỗi chạy
cau_hinh = L.tai_cau_hinh(f"{DUONG_DAN}/configs/experiments.json")
if L.chay_thi_nghiem("E_main", cau_hinh, epochs=1, ten="thu_E_main"):
    L.tom_tat("thu_E_main")

In [ ]:
# Bước 7: chạy thử 1 epoch với baseline (E0). Cần điền "cfg" của E0 trong configs/experiments.json, push lên GitHub, rồi chạy lại Bước 1.
if L.chay_thi_nghiem("E0", cau_hinh, epochs=1, ten="thu_E0"):
    L.tom_tat("thu_E0")

In [ ]:
# Bước 8: chạy thử 1 epoch với cấu hình đã bỏ module (E2), làm tương tự Bước 7
if L.chay_thi_nghiem("E2", cau_hinh, epochs=1, ten="thu_E2"):
    L.tom_tat("thu_E2")